# Evaluación Comparativa de Dimensionalidad y Balanceo de Clases

Este notebook tiene como objetivo evaluar metodológicamente y comparar de forma cuantitativa el impacto de:
1. **Reducción de Dimensionalidad**: Comparar el uso del **Top 30 Híbrido** de variables de consenso, el **Top 10 Híbrido** y una reducción multivariante mediante **MCA (Análisis de Correspondencias Múltiples)** con 10 componentes.
2. **Técnicas de Balanceo de Clases**: Evaluar el desempeño bajo **Baseline (sin balanceo)**, **SMOTE** y **SMOTETomek** para mitigar el desbalance en los niveles de riesgo y tipos de violencia.

Los modelos evaluados son:
* **Random Forest (RF)**
* **XGBoost (XGB)**
* **LightGBM (LGBM)**
* **CatBoost (CatBoost)**

Se evalúan de manera robusta usando las semillas `42`, `52` y `62` de forma sistemática y un tamaño de muestra configurable (`SAMPLE_SIZE`) para evitar desbordamiento de memoria (OOM) en entornos de 16GB RAM.

In [1]:
# Instalar dependencias faltantes de forma condicional y a prueba de fallos
import sys
libs_to_check = {
    'xgboost': 'xgboost',
    'lightgbm': 'lightgbm',
    'catboost': 'catboost',
    'prince': 'prince',
    'imblearn': 'imbalanced-learn',
    'sklearn': 'scikit-learn',
    'tabulate': 'tabulate'
}

libs_to_install = []
for lib_import, lib_install in libs_to_check.items():
    try:
        __import__(lib_import)
    except ImportError:
        libs_to_install.append(lib_install)

if libs_to_install:
    print(f"Instalando dependencias necesarias: {libs_to_install}...")
    !{sys.executable} -m pip install {' '.join(libs_to_install)}
else:
    print("Todas las dependencias principales están instaladas en el kernel activo.")

Todas las dependencias principales están instaladas en el kernel activo.


In [2]:
import gc
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import prince
from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 1000)

## Carga de Datos y Parámetros Globales
Cargamos la base de modelado Parquet y los rankings de consenso híbrido de cada target.

In [ ]:
# Parámetros configurables
SAMPLE_SIZE = 400000
SEEDS = [42, 52, 62]
BASE_PATH = Path("/home/munasqa/MAESTRIA/opencode")

print("Cargando dataset base de modelado...")
df_base = pd.read_parquet(BASE_PATH / "base_modelado.parquet")

if "nivel_de_riesgo_victima" in df_base.columns and "nivel_riesgo_victima" not in df_base.columns:
    df_base = df_base.rename(columns={"nivel_de_riesgo_victima": "nivel_riesgo_victima"})

print(f"Dataset base cargado con dimensiones: {df_base.shape}")

Cargando dataset base de modelado...
Dataset base cargado con dimensiones: (932860, 131)


## Módulos de Entrenamiento y Auditoría de Leakage
Definimos funciones genéricas para procesar los datos de manera limpia, evitando fuga de información (leakage) y entrenando clasificadores robustos.

In [4]:
def get_clean_features(df, target, consensus_features):
    """
    Genera el subconjunto de características libre de leakage.
    """
    leakage_patterns = [
        'tipo_violencia', 'nivel_riesgo_victima', 
        'nivel_de_riesgo_victima', '_orig', '_lbl'
    ]
    
    clean_list = []
    for f in consensus_features:
        # Evitar leak directo
        if any(p in f for p in leakage_patterns) and f != target:
            continue
        clean_list.append(f)
        
    return clean_list

def to_ordinal_encoded_split(X_train_df, X_test_df):
    """
    Aplica OrdinalEncoder robusto ajustando sobre train y transformando train y test.
    Previene errores de tipo de dato string ('desconocido') en los clasificadores.
    """
    X_train = X_train_df.copy()
    X_test = X_test_df.copy()
    
    cat_cols = []
    num_cols = []
    for c in X_train.columns:
        if pd.api.types.is_numeric_dtype(X_train[c]) and not pd.api.types.is_bool_dtype(X_train[c]):
            num_cols.append(c)
        else:
            cat_cols.append(c)
    
    if cat_cols:
        # Asegurar tipo string para el encoder
        X_train[cat_cols] = X_train[cat_cols].astype(str)
        X_test[cat_cols] = X_test[cat_cols].astype(str)
        
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols]).astype(np.float32)
        X_test[cat_cols] = enc.transform(X_test[cat_cols]).astype(np.float32)
        
    for c in num_cols:
        med = pd.to_numeric(X_train[c], errors="coerce").median()
        fill_val = 0.0 if pd.isna(med) else med
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce").fillna(fill_val).astype(np.float32)
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").fillna(fill_val).astype(np.float32)
        
    # Relleno de seguridad final para cualquier NaN remanente
    X_train = X_train.fillna(0.0)
    X_test = X_test.fillna(0.0)
    
    return X_train, X_test

def get_models(seed):
    """
    Instancia los 4 clasificadores con la semilla especificada.
    """
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100, max_depth=12, random_state=seed, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1, 
            eval_metric='mlogloss', verbosity=0
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1, 
            verbosity=-1
        ),
        "CatBoost": CatBoostClassifier(
            iterations=100, depth=6, random_state=seed, verbose=0, thread_count=-1
        )
    }

def color_semaphore(val):
    """
    Asigna un emoji de semáforo según el F1-macro para visualización.
    """
    if val >= 0.75:
        return f"🟢 {val:.4f}"
    elif val >= 0.60:
        return f"🟡 {val:.4f}"
    else:
        return f"🔴 {val:.4f}"

## FASE 1: Reducción de Dimensionalidad - `tipo_violencia`
Evaluamos el desempeño del Top 30, Top 10 y MCA (10 componentes) bajo las tres semillas.

In [5]:
target = "tipo_violencia"
print(f"=== FASE 1: Entrenando Target '{target}' ===")

# 1. Cargar features de consenso híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)
features_top10 = features_top30[:10]

# 2. Muestreo estratificado
df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

results = []

# 3. Iterar por semillas
for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Preprocesar MCA sobre el Top 30
    # MCA requiere que las variables de entrada se traten como string (categóricas)
    X_train_mca_str = X_train_b.astype(str)
    X_test_mca_str = X_test_b.astype(str)
    
    mca = prince.MCA(n_components=10, n_iter=3, random_state=seed, check_input=True)
    X_train_mca = mca.fit_transform(X_train_mca_str)
    X_test_mca = mca.transform(X_test_mca_str)
    
    # Codificar variables categóricas (para clasificadores en Top 30 y Top 10)
    X_train_enc, X_test_enc = to_ordinal_encoded_split(X_train_b, X_test_b)
    
    # Clasificadores
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # Configuración A: Top 30
        clf.fit(X_train_enc, y_train)
        preds = clf.predict(X_test_enc)
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "Top 30 Híbrido",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # Configuración B: Top 10
        clf.fit(X_train_enc[features_top10], y_train)
        preds = clf.predict(X_test_enc[features_top10])
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "Top 10 Híbrido",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # Configuración C: MCA (10 componentes)
        clf.fit(X_train_mca, y_train)
        preds = clf.predict(X_test_mca)
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "MCA (10 Comp)",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        del clf
        gc.collect()
        
    del mca, X_train_enc, X_test_enc, X_train_mca, X_test_mca
    gc.collect()

# 4. Generar Reporte de Consolidación
df_res = pd.DataFrame(results)

# 4.1 Tabla de Estabilidad por Semilla (F1-macro)
df_stability = df_res.pivot_table(
    index=["Modelo", "Esquema"], 
    columns="Semilla", 
    values="F1_macro"
).reset_index()

print(f"\n--- ESTABILIDAD DE F1-MACRO POR SEMILLA: {target.upper()} ---")
from IPython.display import Markdown, display
display(Markdown(df_stability.to_markdown(index=False)))

# 4.2 Tabla Consolidada Multimétrica (Media ± Desviación Estándar)
metrics = ["Accuracy", "F1_macro", "Precision_macro", "Recall_macro"]
df_stats = df_res.groupby(["Modelo", "Esquema"])[metrics].agg(["mean", "std"]).reset_index()

df_report = pd.DataFrame()
df_report["Modelo"] = df_stats["Modelo"]
df_report["Esquema"] = df_stats["Esquema"]

for metric in metrics:
    mean_val = df_stats[metric]["mean"]
    std_val = df_stats[metric]["std"]
    if metric == "F1_macro":
        df_report[metric] = [color_semaphore(m) + f" ± {s:.4f}" for m, s in zip(mean_val, std_val)]
    else:
        df_report[metric] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean_val, std_val)]

print(f"\n--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: {target.upper()} (Media ± Desviación Estándar) ---")
display(Markdown(df_report.to_markdown(index=False)))

# Guardar resultados de Fase 1 para persistencia
df_res.to_csv(BASE_PATH / f'benchmark_reduccion_semillas_{target}.csv', index=False)

=== FASE 1: Entrenando Target 'tipo_violencia' ===
  -> Procesando semilla 42...
  -> Procesando semilla 52...
  -> Procesando semilla 62...

--- ESTABILIDAD DE F1-MACRO POR SEMILLA: TIPO_VIOLENCIA ---


| Modelo        | Esquema        |       42 |       52 |       62 |
|:--------------|:---------------|---------:|---------:|---------:|
| CatBoost      | MCA (10 Comp)  | 0.950006 | 0.943583 | 0.944647 |
| CatBoost      | Top 10 Híbrido | 0.916587 | 0.917165 | 0.912968 |
| CatBoost      | Top 30 Híbrido | 0.982485 | 0.981175 | 0.979963 |
| LightGBM      | MCA (10 Comp)  | 0.950906 | 0.945638 | 0.945463 |
| LightGBM      | Top 10 Híbrido | 0.916587 | 0.917165 | 0.912968 |
| LightGBM      | Top 30 Híbrido | 0.982176 | 0.981618 | 0.980001 |
| Random Forest | MCA (10 Comp)  | 0.945879 | 0.942705 | 0.944555 |
| Random Forest | Top 10 Híbrido | 0.916587 | 0.917165 | 0.912968 |
| Random Forest | Top 30 Híbrido | 0.965028 | 0.968334 | 0.9694   |
| XGBoost       | MCA (10 Comp)  | 0.958367 | 0.955957 | 0.954377 |
| XGBoost       | Top 10 Híbrido | 0.916587 | 0.917228 | 0.912968 |
| XGBoost       | Top 30 Híbrido | 0.982693 | 0.981598 | 0.980573 |


--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: TIPO_VIOLENCIA (Media ± Desviación Estándar) ---


| Modelo        | Esquema        | Accuracy        | F1_macro           | Precision_macro   | Recall_macro    |
|:--------------|:---------------|:----------------|:-------------------|:------------------|:----------------|
| CatBoost      | MCA (10 Comp)  | 0.9487 ± 0.0036 | 🟢 0.9461 ± 0.0034 | 0.9480 ± 0.0035   | 0.9444 ± 0.0033 |
| CatBoost      | Top 10 Híbrido | 0.9244 ± 0.0020 | 🟢 0.9156 ± 0.0023 | 0.9071 ± 0.0022   | 0.9291 ± 0.0022 |
| CatBoost      | Top 30 Híbrido | 0.9856 ± 0.0011 | 🟢 0.9812 ± 0.0013 | 0.9836 ± 0.0017   | 0.9790 ± 0.0009 |
| LightGBM      | MCA (10 Comp)  | 0.9499 ± 0.0034 | 🟢 0.9473 ± 0.0031 | 0.9499 ± 0.0027   | 0.9452 ± 0.0034 |
| LightGBM      | Top 10 Híbrido | 0.9244 ± 0.0020 | 🟢 0.9156 ± 0.0023 | 0.9071 ± 0.0022   | 0.9291 ± 0.0022 |
| LightGBM      | Top 30 Híbrido | 0.9857 ± 0.0010 | 🟢 0.9813 ± 0.0011 | 0.9841 ± 0.0015   | 0.9786 ± 0.0008 |
| Random Forest | MCA (10 Comp)  | 0.9472 ± 0.0018 | 🟢 0.9444 ± 0.0016 | 0.9504 ± 0.0014   | 0.9395 ± 0.0017 |
| Random Forest | Top 10 Híbrido | 0.9244 ± 0.0020 | 🟢 0.9156 ± 0.0023 | 0.9071 ± 0.0022   | 0.9291 ± 0.0022 |
| Random Forest | Top 30 Híbrido | 0.9727 ± 0.0025 | 🟢 0.9676 ± 0.0023 | 0.9758 ± 0.0013   | 0.9606 ± 0.0030 |
| XGBoost       | MCA (10 Comp)  | 0.9594 ± 0.0024 | 🟢 0.9562 ± 0.0020 | 0.9588 ± 0.0020   | 0.9539 ± 0.0020 |
| XGBoost       | Top 10 Híbrido | 0.9244 ± 0.0020 | 🟢 0.9156 ± 0.0023 | 0.9072 ± 0.0023   | 0.9292 ± 0.0022 |
| XGBoost       | Top 30 Híbrido | 0.9859 ± 0.0009 | 🟢 0.9816 ± 0.0011 | 0.9840 ± 0.0014   | 0.9793 ± 0.0008 |

## FASE 1: Reducción de Dimensionalidad - `nivel_riesgo_victima`
Evaluamos el desempeño del Top 30, Top 10 y MCA (10 componentes) bajo las tres semillas.

In [6]:
target = "nivel_riesgo_victima"
print(f"=== FASE 1: Entrenando Target '{target}' ===")

# 1. Cargar features de consenso híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)
features_top10 = features_top30[:10]

# 2. Muestreo estratificado
df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

results = []

# 3. Iterar por semillas
for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Preprocesar MCA sobre el Top 30
    X_train_mca_str = X_train_b.astype(str)
    X_test_mca_str = X_test_b.astype(str)
    
    mca = prince.MCA(n_components=10, n_iter=3, random_state=seed, check_input=True)
    X_train_mca = mca.fit_transform(X_train_mca_str)
    X_test_mca = mca.transform(X_test_mca_str)
    
    # Codificar variables categóricas (para clasificadores en Top 30 y Top 10)
    X_train_enc, X_test_enc = to_ordinal_encoded_split(X_train_b, X_test_b)
    
    # Clasificadores
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # Configuración A: Top 30
        clf.fit(X_train_enc, y_train)
        preds = clf.predict(X_test_enc)
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "Top 30 Híbrido",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # Configuración B: Top 10
        clf.fit(X_train_enc[features_top10], y_train)
        preds = clf.predict(X_test_enc[features_top10])
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "Top 10 Híbrido",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # Configuración C: MCA (10 componentes)
        clf.fit(X_train_mca, y_train)
        preds = clf.predict(X_test_mca)
        results.append({
            "Semilla": seed, "Modelo": model_name, "Esquema": "MCA (10 Comp)",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        del clf
        gc.collect()
        
    del mca, X_train_enc, X_test_enc, X_train_mca, X_test_mca
    gc.collect()

# 4. Generar Reporte de Consolidación
df_res = pd.DataFrame(results)

# 4.1 Tabla de Estabilidad por Semilla (F1-macro)
df_stability = df_res.pivot_table(
    index=["Modelo", "Esquema"], 
    columns="Semilla", 
    values="F1_macro"
).reset_index()

print(f"\n--- ESTABILIDAD DE F1-MACRO POR SEMILLA: {target.upper()} ---")
display(Markdown(df_stability.to_markdown(index=False)))

# 4.2 Tabla Consolidada Multimétrica (Media ± Desviación Estándar)
metrics = ["Accuracy", "F1_macro", "Precision_macro", "Recall_macro"]
df_stats = df_res.groupby(["Modelo", "Esquema"])[metrics].agg(["mean", "std"]).reset_index()

df_report = pd.DataFrame()
df_report["Modelo"] = df_stats["Modelo"]
df_report["Esquema"] = df_stats["Esquema"]

for metric in metrics:
    mean_val = df_stats[metric]["mean"]
    std_val = df_stats[metric]["std"]
    if metric == "F1_macro":
        df_report[metric] = [color_semaphore(m) + f" ± {s:.4f}" for m, s in zip(mean_val, std_val)]
    else:
        df_report[metric] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean_val, std_val)]

print(f"\n--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: {target.upper()} (Media ± Desviación Estándar) ---")
display(Markdown(df_report.to_markdown(index=False)))

# Guardar resultados de Fase 1 para persistencia
df_res.to_csv(BASE_PATH / f'benchmark_reduccion_semillas_{target}.csv', index=False)

=== FASE 1: Entrenando Target 'nivel_riesgo_victima' ===
  -> Procesando semilla 42...
  -> Procesando semilla 52...
  -> Procesando semilla 62...

--- ESTABILIDAD DE F1-MACRO POR SEMILLA: NIVEL_RIESGO_VICTIMA ---


| Modelo        | Esquema        |       42 |       52 |       62 |
|:--------------|:---------------|---------:|---------:|---------:|
| CatBoost      | MCA (10 Comp)  | 0.437362 | 0.446098 | 0.452902 |
| CatBoost      | Top 10 Híbrido | 0.488092 | 0.50062  | 0.497193 |
| CatBoost      | Top 30 Híbrido | 0.558019 | 0.557943 | 0.560064 |
| LightGBM      | MCA (10 Comp)  | 0.421196 | 0.400112 | 0.417809 |
| LightGBM      | Top 10 Híbrido | 0.478357 | 0.474837 | 0.483712 |
| LightGBM      | Top 30 Híbrido | 0.53589  | 0.538791 | 0.540527 |
| Random Forest | MCA (10 Comp)  | 0.423409 | 0.411816 | 0.431336 |
| Random Forest | Top 10 Híbrido | 0.41992  | 0.421569 | 0.427945 |
| Random Forest | Top 30 Híbrido | 0.449064 | 0.456219 | 0.458988 |
| XGBoost       | MCA (10 Comp)  | 0.468804 | 0.470694 | 0.468643 |
| XGBoost       | Top 10 Híbrido | 0.51281  | 0.520168 | 0.516942 |
| XGBoost       | Top 30 Híbrido | 0.573524 | 0.574923 | 0.574718 |


--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: NIVEL_RIESGO_VICTIMA (Media ± Desviación Estándar) ---


| Modelo        | Esquema        | Accuracy        | F1_macro           | Precision_macro   | Recall_macro    |
|:--------------|:---------------|:----------------|:-------------------|:------------------|:----------------|
| CatBoost      | MCA (10 Comp)  | 0.5629 ± 0.0058 | 🔴 0.4455 ± 0.0078 | 0.5583 ± 0.0104   | 0.4427 ± 0.0063 |
| CatBoost      | Top 10 Híbrido | 0.5900 ± 0.0021 | 🔴 0.4953 ± 0.0065 | 0.6063 ± 0.0037   | 0.4823 ± 0.0046 |
| CatBoost      | Top 30 Híbrido | 0.6187 ± 0.0004 | 🔴 0.5587 ± 0.0012 | 0.6330 ± 0.0009   | 0.5366 ± 0.0012 |
| LightGBM      | MCA (10 Comp)  | 0.5621 ± 0.0032 | 🔴 0.4130 ± 0.0113 | 0.5865 ± 0.0081   | 0.4230 ± 0.0069 |
| LightGBM      | Top 10 Híbrido | 0.5886 ± 0.0009 | 🔴 0.4790 ± 0.0045 | 0.6193 ± 0.0015   | 0.4703 ± 0.0030 |
| LightGBM      | Top 30 Híbrido | 0.6135 ± 0.0014 | 🔴 0.5384 ± 0.0023 | 0.6411 ± 0.0010   | 0.5172 ± 0.0023 |
| Random Forest | MCA (10 Comp)  | 0.5694 ± 0.0041 | 🔴 0.4222 ± 0.0098 | 0.6159 ± 0.0110   | 0.4299 ± 0.0063 |
| Random Forest | Top 10 Híbrido | 0.5729 ± 0.0012 | 🔴 0.4231 ± 0.0042 | 0.6315 ± 0.0034   | 0.4335 ± 0.0019 |
| Random Forest | Top 30 Híbrido | 0.5850 ± 0.0031 | 🔴 0.4548 ± 0.0051 | 0.6477 ± 0.0038   | 0.4526 ± 0.0039 |
| XGBoost       | MCA (10 Comp)  | 0.5727 ± 0.0025 | 🔴 0.4694 ± 0.0011 | 0.5740 ± 0.0066   | 0.4607 ± 0.0011 |
| XGBoost       | Top 10 Híbrido | 0.5976 ± 0.0012 | 🔴 0.5166 ± 0.0037 | 0.6083 ± 0.0044   | 0.4997 ± 0.0030 |
| XGBoost       | Top 30 Híbrido | 0.6294 ± 0.0006 | 🔴 0.5744 ± 0.0008 | 0.6445 ± 0.0007   | 0.5514 ± 0.0009 |

## FASE 2: Comparativa de Técnicas de Balanceo de Categorías
Una vez comprobada la solidez y consistencia del **Top 30 Híbrido por Consenso** (con el mejor desempeño promedio en las tres semillas), lo utilizamos para evaluar los escenarios de balanceo:
1. **Baseline**: Sin balanceo.
2. **SMOTE** (Synthetic Minority Over-sampling Technique).
3. **SMOTETomek** (SMOTE + Tomek Links de remoción de ruido).

Se evalúan bajo los mismos clasificadores (RF, XGBoost, LightGBM, CatBoost) y semillas (`42`, `52`, `62`).

### FASE 2: Balanceo de Categorías - `tipo_violencia`

In [7]:
target = "tipo_violencia"
print(f"=== FASE 2: Balanceo para Target '{target}' ===")

# Cargar variables del Top 30 Híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)

df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

bal_results = []

for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Codificar variables string/categóricas antes del balanceo
    X_train_enc, X_test_enc = to_ordinal_encoded_split(X_train_b, X_test_b)
    
    # Instanciar balanceadores
    smote = SMOTE(random_state=seed)
    smotetomek = SMOTETomek(random_state=seed)
    
    # Aplicar balanceadores sobre el conjunto de train codificado
    X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train)
    X_train_tomek, y_train_tomek = smotetomek.fit_resample(X_train_enc, y_train)
    
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # 1. Escenario Baseline
        clf.fit(X_train_enc, y_train)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "Baseline",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # 2. Escenario SMOTE
        clf.fit(X_train_smote, y_train_smote)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "SMOTE",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # 3. Escenario SMOTETomek
        clf.fit(X_train_tomek, y_train_tomek)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "SMOTETomek",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        del clf
        gc.collect()
        
    del smote, smotetomek, X_train_enc, X_test_enc, X_train_smote, y_train_smote, X_train_tomek, y_train_tomek
    gc.collect()

# Consolidar y Reportar
df_bal = pd.DataFrame(bal_results)

# 4.1 Tabla de Estabilidad por Semilla (F1-macro)
df_stability = df_bal.pivot_table(
    index=["Modelo", "Balanceo"], 
    columns="Semilla", 
    values="F1_macro"
).reset_index()

print(f"\n--- ESTABILIDAD DE F1-MACRO POR SEMILLA: {target.upper()} ---")
from IPython.display import Markdown, display
display(Markdown(df_stability.to_markdown(index=False)))

# 4.2 Tabla Consolidada Multimétrica (Media ± Desviación Estándar)
metrics = ["Accuracy", "F1_macro", "Precision_macro", "Recall_macro"]
df_stats = df_bal.groupby(["Modelo", "Balanceo"])[metrics].agg(["mean", "std"]).reset_index()

df_report = pd.DataFrame()
df_report["Modelo"] = df_stats["Modelo"]
df_report["Balanceo"] = df_stats["Balanceo"]

for metric in metrics:
    mean_val = df_stats[metric]["mean"]
    std_val = df_stats[metric]["std"]
    if metric == "F1_macro":
        df_report[metric] = [color_semaphore(m) + f" ± {s:.4f}" for m, s in zip(mean_val, std_val)]
    else:
        df_report[metric] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean_val, std_val)]

print(f"\n--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: {target.upper()} (Media ± Desviación Estándar) ---")
display(Markdown(df_report.to_markdown(index=False)))

df_bal.to_csv(BASE_PATH / f"benchmark_balanceo_semillas_{target}.csv", index=False)

=== FASE 2: Balanceo para Target 'tipo_violencia' ===
  -> Procesando semilla 42...
  -> Procesando semilla 52...
  -> Procesando semilla 62...

--- ESTABILIDAD DE F1-MACRO POR SEMILLA: TIPO_VIOLENCIA ---


| Modelo        | Balanceo   |       42 |       52 |       62 |
|:--------------|:-----------|---------:|---------:|---------:|
| CatBoost      | Baseline   | 0.982485 | 0.981175 | 0.979963 |
| CatBoost      | SMOTE      | 0.981747 | 0.980993 | 0.978571 |
| CatBoost      | SMOTETomek | 0.982163 | 0.981397 | 0.978343 |
| LightGBM      | Baseline   | 0.982176 | 0.981618 | 0.980001 |
| LightGBM      | SMOTE      | 0.982253 | 0.980155 | 0.978529 |
| LightGBM      | SMOTETomek | 0.982177 | 0.979881 | 0.978456 |
| Random Forest | Baseline   | 0.965028 | 0.968334 | 0.9694   |
| Random Forest | SMOTE      | 0.968724 | 0.968924 | 0.967819 |
| Random Forest | SMOTETomek | 0.970343 | 0.969547 | 0.965613 |
| XGBoost       | Baseline   | 0.982693 | 0.981598 | 0.980573 |
| XGBoost       | SMOTE      | 0.982438 | 0.981102 | 0.979303 |
| XGBoost       | SMOTETomek | 0.982519 | 0.981304 | 0.979713 |


--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: TIPO_VIOLENCIA (Media ± Desviación Estándar) ---


| Modelo        | Balanceo   | Accuracy        | F1_macro           | Precision_macro   | Recall_macro    |
|:--------------|:-----------|:----------------|:-------------------|:------------------|:----------------|
| CatBoost      | Baseline   | 0.9856 ± 0.0011 | 🟢 0.9812 ± 0.0013 | 0.9836 ± 0.0017   | 0.9790 ± 0.0009 |
| CatBoost      | SMOTE      | 0.9850 ± 0.0014 | 🟢 0.9804 ± 0.0017 | 0.9809 ± 0.0020   | 0.9800 ± 0.0013 |
| CatBoost      | SMOTETomek | 0.9851 ± 0.0016 | 🟢 0.9806 ± 0.0020 | 0.9810 ± 0.0026   | 0.9802 ± 0.0015 |
| LightGBM      | Baseline   | 0.9857 ± 0.0010 | 🟢 0.9813 ± 0.0011 | 0.9841 ± 0.0015   | 0.9786 ± 0.0008 |
| LightGBM      | SMOTE      | 0.9849 ± 0.0015 | 🟢 0.9803 ± 0.0019 | 0.9801 ± 0.0020   | 0.9806 ± 0.0017 |
| LightGBM      | SMOTETomek | 0.9848 ± 0.0015 | 🟢 0.9802 ± 0.0019 | 0.9800 ± 0.0021   | 0.9803 ± 0.0017 |
| Random Forest | Baseline   | 0.9727 ± 0.0025 | 🟢 0.9676 ± 0.0023 | 0.9758 ± 0.0013   | 0.9606 ± 0.0030 |
| Random Forest | SMOTE      | 0.9729 ± 0.0004 | 🟢 0.9685 ± 0.0006 | 0.9691 ± 0.0011   | 0.9681 ± 0.0003 |
| Random Forest | SMOTETomek | 0.9732 ± 0.0021 | 🟢 0.9685 ± 0.0025 | 0.9691 ± 0.0028   | 0.9681 ± 0.0022 |
| XGBoost       | Baseline   | 0.9859 ± 0.0009 | 🟢 0.9816 ± 0.0011 | 0.9840 ± 0.0014   | 0.9793 ± 0.0008 |
| XGBoost       | SMOTE      | 0.9853 ± 0.0013 | 🟢 0.9809 ± 0.0016 | 0.9814 ± 0.0018   | 0.9805 ± 0.0014 |
| XGBoost       | SMOTETomek | 0.9855 ± 0.0012 | 🟢 0.9812 ± 0.0014 | 0.9818 ± 0.0017   | 0.9806 ± 0.0011 |

### FASE 2: Balanceo de Categorías - `nivel_riesgo_victima`

In [8]:
target = "nivel_riesgo_victima"
print(f"=== FASE 2: Balanceo para Target '{target}' ===")

# Cargar variables del Top 30 Híbrido
df_consensus = pd.read_csv(BASE_PATH / f"top30_consenso_{target}.csv")
features_top30 = df_consensus["feature"].head(30).tolist()
features_top30 = get_clean_features(df_base, target, features_top30)

df_sample = df_base.sample(n=min(SAMPLE_SIZE, len(df_base)), random_state=42).copy()
X_base = df_sample[features_top30]
y = df_sample[target].astype(int)

bal_results = []

for seed in SEEDS:
    print(f"  -> Procesando semilla {seed}...")
    # Splits
    X_train_b, X_test_b, y_train, y_test = train_test_split(
        X_base, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    # Codificar variables string/categóricas antes del balanceo
    X_train_enc, X_test_enc = to_ordinal_encoded_split(X_train_b, X_test_b)
    
    # Instanciar balanceadores
    smote = SMOTE(random_state=seed)
    smotetomek = SMOTETomek(random_state=seed)
    
    # Aplicar balanceadores sobre el conjunto de train codificado
    X_train_smote, y_train_smote = smote.fit_resample(X_train_enc, y_train)
    X_train_tomek, y_train_tomek = smotetomek.fit_resample(X_train_enc, y_train)
    
    models = get_models(seed)
    
    for model_name, clf in models.items():
        # 1. Escenario Baseline
        clf.fit(X_train_enc, y_train)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "Baseline",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # 2. Escenario SMOTE
        clf.fit(X_train_smote, y_train_smote)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "SMOTE",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        # 3. Escenario SMOTETomek
        clf.fit(X_train_tomek, y_train_tomek)
        preds = clf.predict(X_test_enc)
        bal_results.append({
            "Semilla": seed, "Modelo": model_name, "Balanceo": "SMOTETomek",
            "Accuracy": accuracy_score(y_test, preds),
            "F1_macro": f1_score(y_test, preds, average='macro'),
            "Precision_macro": precision_score(y_test, preds, average='macro'),
            "Recall_macro": recall_score(y_test, preds, average='macro')
        })
        
        del clf
        gc.collect()
        
    del smote, smotetomek, X_train_enc, X_test_enc, X_train_smote, y_train_smote, X_train_tomek, y_train_tomek
    gc.collect()

# Consolidar y Reportar
df_bal = pd.DataFrame(bal_results)

# 4.1 Tabla de Estabilidad por Semilla (F1-macro)
df_stability = df_bal.pivot_table(
    index=["Modelo", "Balanceo"], 
    columns="Semilla", 
    values="F1_macro"
).reset_index()

print(f"\n--- ESTABILIDAD DE F1-MACRO POR SEMILLA: {target.upper()} ---")
from IPython.display import Markdown, display
display(Markdown(df_stability.to_markdown(index=False)))

# 4.2 Tabla Consolidada Multimétrica (Media ± Desviación Estándar)
metrics = ["Accuracy", "F1_macro", "Precision_macro", "Recall_macro"]
df_stats = df_bal.groupby(["Modelo", "Balanceo"])[metrics].agg(["mean", "std"]).reset_index()

df_report = pd.DataFrame()
df_report["Modelo"] = df_stats["Modelo"]
df_report["Balanceo"] = df_stats["Balanceo"]

for metric in metrics:
    mean_val = df_stats[metric]["mean"]
    std_val = df_stats[metric]["std"]
    if metric == "F1_macro":
        df_report[metric] = [color_semaphore(m) + f" ± {s:.4f}" for m, s in zip(mean_val, std_val)]
    else:
        df_report[metric] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean_val, std_val)]

print(f"\n--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: {target.upper()} (Media ± Desviación Estándar) ---")
display(Markdown(df_report.to_markdown(index=False)))

df_bal.to_csv(BASE_PATH / f"benchmark_balanceo_semillas_{target}.csv", index=False)

=== FASE 2: Balanceo para Target 'nivel_riesgo_victima' ===
  -> Procesando semilla 42...
  -> Procesando semilla 52...
  -> Procesando semilla 62...

--- ESTABILIDAD DE F1-MACRO POR SEMILLA: NIVEL_RIESGO_VICTIMA ---


| Modelo        | Balanceo   |       42 |       52 |       62 |
|:--------------|:-----------|---------:|---------:|---------:|
| CatBoost      | Baseline   | 0.558019 | 0.557943 | 0.560064 |
| CatBoost      | SMOTE      | 0.578854 | 0.575589 | 0.569315 |
| CatBoost      | SMOTETomek | 0.576684 | 0.571747 | 0.570801 |
| LightGBM      | Baseline   | 0.53589  | 0.538791 | 0.540527 |
| LightGBM      | SMOTE      | 0.559911 | 0.558738 | 0.557016 |
| LightGBM      | SMOTETomek | 0.56491  | 0.559881 | 0.562506 |
| Random Forest | Baseline   | 0.449064 | 0.456219 | 0.458988 |
| Random Forest | SMOTE      | 0.512763 | 0.514435 | 0.507728 |
| Random Forest | SMOTETomek | 0.517952 | 0.51813  | 0.518026 |
| XGBoost       | Baseline   | 0.573524 | 0.574923 | 0.574718 |
| XGBoost       | SMOTE      | 0.586361 | 0.584394 | 0.583457 |
| XGBoost       | SMOTETomek | 0.585316 | 0.584956 | 0.584518 |


--- METRICAS MULTI-OBJETIVO CONSOLIDADAS: NIVEL_RIESGO_VICTIMA (Media ± Desviación Estándar) ---


| Modelo        | Balanceo   | Accuracy        | F1_macro           | Precision_macro   | Recall_macro    |
|:--------------|:-----------|:----------------|:-------------------|:------------------|:----------------|
| CatBoost      | Baseline   | 0.6187 ± 0.0004 | 🔴 0.5587 ± 0.0012 | 0.6330 ± 0.0009   | 0.5366 ± 0.0012 |
| CatBoost      | SMOTE      | 0.6158 ± 0.0045 | 🔴 0.5746 ± 0.0048 | 0.6130 ± 0.0061   | 0.5570 ± 0.0046 |
| CatBoost      | SMOTETomek | 0.6108 ± 0.0026 | 🔴 0.5731 ± 0.0032 | 0.6039 ± 0.0037   | 0.5578 ± 0.0027 |
| LightGBM      | Baseline   | 0.6135 ± 0.0014 | 🔴 0.5384 ± 0.0023 | 0.6411 ± 0.0010   | 0.5172 ± 0.0023 |
| LightGBM      | SMOTE      | 0.6093 ± 0.0018 | 🔴 0.5586 ± 0.0015 | 0.6110 ± 0.0023   | 0.5392 ± 0.0013 |
| LightGBM      | SMOTETomek | 0.6100 ± 0.0020 | 🔴 0.5624 ± 0.0025 | 0.6096 ± 0.0026   | 0.5440 ± 0.0023 |
| Random Forest | Baseline   | 0.5850 ± 0.0031 | 🔴 0.4548 ± 0.0051 | 0.6477 ± 0.0038   | 0.4526 ± 0.0039 |
| Random Forest | SMOTE      | 0.5760 ± 0.0035 | 🔴 0.5116 ± 0.0035 | 0.5713 ± 0.0060   | 0.4964 ± 0.0034 |
| Random Forest | SMOTETomek | 0.5747 ± 0.0038 | 🔴 0.5180 ± 0.0001 | 0.5669 ± 0.0067   | 0.5045 ± 0.0008 |
| XGBoost       | Baseline   | 0.6294 ± 0.0006 | 🔴 0.5744 ± 0.0008 | 0.6445 ± 0.0007   | 0.5514 ± 0.0009 |
| XGBoost       | SMOTE      | 0.6274 ± 0.0014 | 🔴 0.5847 ± 0.0015 | 0.6301 ± 0.0028   | 0.5650 ± 0.0011 |
| XGBoost       | SMOTETomek | 0.6256 ± 0.0002 | 🔴 0.5849 ± 0.0004 | 0.6246 ± 0.0006   | 0.5666 ± 0.0006 |